# SOXL 因子诊断 v1

目标：把 `src/factor` 从 FactorSpec 清单扩展为可复用诊断框架，支持单因子 IC、多因子相关性、分层收益和 IC 加权组合因子。

边界：本 notebook 使用 `src/data` 生成的 supervised 特征作为候选因子；具体可复用计算逻辑在 `src/factor`。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.factor import analyze_factor_suite, build_composite_factor, compute_factor_ic

CARD = REPO_ROOT / 'research/factor_diagnostics_v1'
CARD.mkdir(parents=True, exist_ok=True)

price = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(price, horizon=1)
factor_columns = list(dataset.feature_columns)
dataset.frame[['date', *factor_columns, dataset.target_column]].head()

,date,soxl_return_1d,soxx_return_1d,soxl_return_5d,soxx_return_5d,soxl_soxx_relative_5d,vix_level,vix_change_1d,intraday_range,volume_z_20,trend_5,trend_21,realized_vol_5,realized_vol_21,drawdown_21,target_return
0,2026-01-02,0.000000,0.000000,0.0,0.0,0.0,14.51,0.000000,0.063717,0.0,0.000000,0.0,0.0,0.0,0.0,0.041914
1,2026-01-05,0.041914,0.013931,0.0,0.0,0.0,14.90,0.026878,0.052621,0.0,0.000000,0.0,0.0,0.0,0.0,0.097318
2,2026-01-06,0.097318,0.032352,0.0,0.0,0.0,14.75,-0.010067,0.059804,0.0,0.000000,0.0,0.0,0.0,0.0,-0.031846
3,2026-01-07,-0.031846,-0.010538,0.0,0.0,0.0,15.38,0.042712,0.036336,0.0,0.000000,0.0,0.0,0.0,0.0,-0.050488
4,2026-01-08,-0.050488,-0.016344,0.0,0.0,0.0,15.45,0.004551,0.074119,0.0,-0.016481,0.0,0.0,0.0,0.0,0.086606


In [2]:
report = analyze_factor_suite(
    dataset.frame,
    factor_columns=factor_columns,
    target_column=dataset.target_column,
    quantiles=5,
)
report.ic.to_csv(CARD / 'factor_ic.csv', index=False)
report.correlation.to_csv(CARD / 'factor_correlation.csv', index=False)
report.quantile_returns.to_csv(CARD / 'factor_quantile_returns.csv', index=False)
report.ic.head(10)

,factor,scope,method,ic_mean,ic_std,ic_positive_rate,ic_negative_rate,observation_count,period_count
0,vix_change_1d,time_series,spearman,0.124268,0.0,1.0,0.0,108,1
1,realized_vol_21,time_series,spearman,0.088324,0.0,1.0,0.0,108,1
2,drawdown_21,time_series,spearman,0.074311,0.0,1.0,0.0,108,1
3,vix_level,time_series,spearman,0.060343,0.0,1.0,0.0,108,1
4,realized_vol_5,time_series,spearman,0.058488,0.0,1.0,0.0,108,1
5,intraday_range,time_series,spearman,0.051845,0.0,1.0,0.0,108,1
6,volume_z_20,time_series,spearman,0.044401,0.0,1.0,0.0,108,1
7,trend_21,time_series,spearman,0.039963,0.0,1.0,0.0,108,1
8,soxl_return_5d,time_series,spearman,0.034336,0.0,1.0,0.0,108,1
9,soxl_soxx_relative_5d,time_series,spearman,0.034041,0.0,1.0,0.0,108,1


In [3]:
high_corr = report.correlation.loc[
    (report.correlation['factor_a'] != report.correlation['factor_b'])
    & (report.correlation['abs_correlation'] >= 0.8)
].sort_values('abs_correlation', ascending=False)
high_corr.head(12)

,factor_a,factor_b,method,correlation,abs_correlation
63,soxl_return_1d,soxx_return_1d,spearman,0.999162,0.999162
102,soxx_return_1d,soxl_return_1d,spearman,0.999162,0.999162
76,soxl_return_5d,soxl_soxx_relative_5d,spearman,0.999038,0.999038
89,soxl_soxx_relative_5d,soxl_return_5d,spearman,0.999038,0.999038
78,soxl_return_5d,soxx_return_5d,spearman,0.997170,0.997170
117,soxx_return_5d,soxl_return_5d,spearman,0.997170,0.997170
92,soxl_soxx_relative_5d,soxx_return_5d,spearman,0.994693,0.994693
118,soxx_return_5d,soxl_soxx_relative_5d,spearman,0.994693,0.994693


In [4]:
report.quantile_returns.sort_values(['factor', 'quantile']).head(20)

,factor,quantile,count,mean_return,median_return,spread_vs_bottom
65,drawdown_21,1,22,0.018334,0.023643,0.000000
66,drawdown_21,2,21,0.007685,0.009376,-0.010649
67,drawdown_21,3,22,0.003350,0.027877,-0.014984
68,drawdown_21,4,21,0.013831,0.012975,-0.004503
69,drawdown_21,5,22,0.041025,0.053991,0.022691
35,intraday_range,1,22,0.000766,0.014336,0.000000
36,intraday_range,2,21,0.033935,0.048592,0.033169
37,intraday_range,3,22,0.021254,0.027757,0.020488
38,intraday_range,4,21,0.027371,0.010584,0.026605
39,intraday_range,5,22,0.002708,0.023716,0.001942


In [5]:
composite_weights = {
    str(row.factor): float(row.ic_mean)
    for row in report.ic.itertuples(index=False)
    if abs(float(row.ic_mean)) > 0.05
}
composite = build_composite_factor(
    dataset.frame,
    factor_columns=factor_columns,
    weights=composite_weights,
    output_column='ic_weighted_composite',
)
composite_ic = compute_factor_ic(
    composite,
    factor_columns=['ic_weighted_composite'],
    target_column=dataset.target_column,
)
composite[['date', 'ic_weighted_composite', dataset.target_column]].to_csv(CARD / 'composite_factor.csv', index=False)
composite_ic.to_csv(CARD / 'composite_factor_ic.csv', index=False)
summary = {
    **report.summary,
    'start': str(dataset.frame['date'].min().date()),
    'end': str(dataset.frame['date'].max().date()),
    'row_count': int(len(dataset.frame)),
    'composite_factor': 'ic_weighted_composite',
    'composite_ic_mean': float(composite_ic.iloc[0]['ic_mean']),
    'artifact_files': [
        'factor_ic.csv',
        'factor_correlation.csv',
        'factor_quantile_returns.csv',
        'composite_factor.csv',
        'composite_factor_ic.csv',
    ],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'factor_suite_report.v1',
 'factor_count': 14,
 'target_column': 'target_return',
 'top_factor': 'vix_change_1d',
 'top_ic_mean': 0.12426792293785408,
 'high_correlation_pair_count': 8,
 'review_text': '## 中文结论\n\n- 本次评估 14 个因子，目标列为 `target_return`。\n- IC 最强因子为 `vix_change_1d`，IC 均值为 0.1243。\n- 高相关因子对数量为 8，用于提示多因子冗余风险。\n- 分层收益已输出 quantile 表，用于判断因子方向和单调性。',
 'start': '2026-01-02',
 'end': '2026-06-08',
 'row_count': 108,
 'composite_factor': 'ic_weighted_composite',
 'composite_ic_mean': 0.03079062943591795,
 'artifact_files': ['factor_ic.csv',
  'factor_correlation.csv',
  'factor_quantile_returns.csv',
  'composite_factor.csv',
  'composite_factor_ic.csv']}

## 中文结论

- 做了什么：新增 `src/factor` 可复用诊断能力，并在本 notebook 中评估 14 个 supervised 候选因子。
- 核心指标：IC 最强因子为 `vix_change_1d`，IC 均值 0.1243；IC 加权组合因子 IC 均值 0.0308。
- 多因子差异：高相关因子对数量为 8，后续训练前需要去冗余或降权。
- 分层收益：`factor_quantile_returns.csv` 已输出各因子的 quantile 平均收益和相对底层 spread，用于检查方向与单调性。
- regime 差异：本 notebook 不额外引入新的 regime 结论；如有分层结果，以已写出的 analysis/regime/window sidecar 为准。
- 是否晋升：否。本轮是框架能力验证和候选因子诊断，不直接晋升 production。
- 风险：当前是 SOXL 单资产时间序列 IC；未来扩展多资产 universe 时应切到横截面 IC，并在 `entity_column` 下按日期聚合。
- 下一步：把通过 IC、相关性和分层收益筛选的因子接入模型矩阵，比较单因子、多因子、模型特征集的回测差异。